In [1]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

import json
import numpy as np
import pandas as pd
from IPython.display import display

results_dir = RESULTS_DIR
if not results_dir.is_dir():
    raise FileNotFoundError(f"Missing judge results: {results_dir}")

In [2]:
models = ["gpt", "claude", "gemini", "deepseek"]
techniques = ["zero_shot", "few_shot", "mv3"]

files = {
    (model, technique): results_dir / f"{model}_{technique}" / (
        "classifications.jsonl" if technique == "mv3" else "responses.jsonl"
    )
    for model in models
    for technique in techniques
}

for path in files.values():
    assert path.is_file(), f"File not found: {path}"

In [3]:
records = []

for (model, technique), path in files.items():
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                records.append({
                    "model": model,
                    "technique": technique,
                    "result": record.get("result", record),
                })

pd.DataFrame(records).groupby(
    ["model", "technique"]
).size().unstack("technique")

technique,few_shot,mv3,zero_shot
model,,,
claude,259,257,259
deepseek,260,257,260
gemini,259,259,260
gpt,260,260,260


In [4]:
evaluations = pd.DataFrame([
    {
        "model": r["model"],
        "technique": r["technique"],
        "prompt_id": r["result"].get("prompt_id"),
        "has_smells": isinstance(r["result"].get("smells"), dict),
    }
    for r in records
])

assert evaluations["prompt_id"].notna().all(), "Missing prompt IDs."
assert evaluations["has_smells"].all(), "Missing or invalid smell results."
assert not evaluations.duplicated(
    ["model", "technique", "prompt_id"]
).any(), "Duplicate evaluations found."

In [5]:
syntactic = [
    "grammar_smell",
    "formatting_smell",
    "readability_smell",
    "excessive_prompt_length",
    "poor_prompt_organization",
    "role_separation_smell",
]

semantic = [
    "ambiguous_instruction",
    "missing_constraints",
    "conflicting_instructions",
    "intent_misalignment",
    "missing_context_smell",
    "irrelevant_context",
    "complexity_smell",
    "inefficient_few_shot_examples",
    "overloaded_prompt",
    "undefined_output_format",
    "incorrect_information",
    "prompt_injection",
    "unsafe_prompt",
    "bias_smell",
    "style_formality_mismatch",
]

categories = {
    **dict.fromkeys(syntactic, "Syntactic/Structural"),
    **dict.fromkeys(semantic, "Semantic"),
}

In [6]:
rows = []

for record in records:
    result = record["result"]

    for smell, annotation in result["smells"].items():
        if smell in categories and annotation["label"] == "present":
            rows.append({
                "model": record["model"],
                "technique": record["technique"],
                "prompt_id": result["prompt_id"],
                "smell": smell,
                "category": categories[smell],
            })

df_present = pd.DataFrame(rows)

In [7]:
prevalence = (
    df_present.groupby(["smell"])
    .size()
    .rename("detections")
    .reset_index()
    .sort_values("detections", ascending=False)
)

prevalence["prevalence_pct"] = (
    prevalence["detections"] / len(evaluations) * 100
).round(1)

prevalence.reset_index(drop=True)

,smell,detections,prevalence_pct
0,ambiguous_instruction,2054,66.0
1,undefined_output_format,1817,58.4
2,missing_constraints,1802,57.9
3,formatting_smell,1447,46.5
4,missing_context_smell,1427,45.9
5,grammar_smell,1204,38.7
6,irrelevant_context,468,15.0
7,poor_prompt_organization,346,11.1
8,conflicting_instructions,301,9.7
9,readability_smell,216,6.9


In [8]:
by_model = pd.crosstab(
    df_present["smell"], df_present["model"]
).reindex(
    index=prevalence["smell"],
    columns=models,
    fill_value=0,
)

by_model

model,gpt,claude,gemini,deepseek
smell,,,,
ambiguous_instruction,544,723,279,508
undefined_output_format,439,512,502,364
missing_constraints,330,665,340,467
formatting_smell,272,617,313,245
missing_context_smell,350,481,246,350
grammar_smell,252,449,167,336
irrelevant_context,126,156,97,89
poor_prompt_organization,26,248,27,45
conflicting_instructions,53,142,47,59


In [9]:
by_technique = pd.crosstab(
    df_present["smell"], df_present["technique"]
).reindex(
    index=prevalence["smell"],
    columns=techniques,
    fill_value=0,
)

by_technique

technique,zero_shot,few_shot,mv3
smell,,,
ambiguous_instruction,668,720,666
undefined_output_format,614,595,608
missing_constraints,585,620,597
formatting_smell,502,468,477
missing_context_smell,474,486,467
grammar_smell,393,415,396
irrelevant_context,152,167,149
poor_prompt_organization,98,150,98
conflicting_instructions,101,103,97


In [ ]:
by_model_technique = pd.crosstab(
    df_present["smell"],
    [df_present["model"], df_present["technique"]],
).reindex(
    index=prevalence["smell"],
    columns=pd.MultiIndex.from_product(
        [models, techniques], names=["model", "technique"]
    ),
    fill_value=0,
)

by_model_technique

In [ ]:
shared_smells = by_model.loc[by_model.gt(0).all(axis=1)].copy()
shared_smells.insert(0, "category", shared_smells.index.map(categories))

display(shared_smells)
display(shared_smells["category"].value_counts().rename("smell_types"))

In [ ]:
types_by_model = (
    df_present.groupby(["model", "category"])["smell"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(models)
)

types_by_model["Total"] = types_by_model.sum(axis=1)

types_by_model

In [ ]:
top_six = pd.DataFrame({
    (model, technique): by_model_technique[(model, technique)]
        .sort_values(ascending=False)
        .head(6)
        .index.tolist()
    for model in models
    for technique in techniques
})

top_six.index = pd.RangeIndex(1, 7, name="rank")
top_six.columns.names = ["model", "technique"]

top_six

In [ ]:
# Build a complete long-format classification matrix (omitted smells are absent).
from smell_catalogue import SMELLS
classifications = pd.DataFrame([
    {
        "model": record["model"],
        "technique": record["technique"],
        "prompt_id": record["result"]["prompt_id"],
        "smell": smell,
        "label": (
            record["result"]["smells"].get(smell, {}).get("label", "absent")
        ),
    }
    for record in records
    for smell in SMELLS
])

coverage = (classifications.groupby(["model", "technique"])["prompt_id"]
            .nunique().unstack().reindex(index=models, columns=techniques))
display(coverage)

# Denominator per model and technique, then sum across techniques if pooled.
per_configuration = (
    classifications.assign(is_present=classifications["label"].eq("present"))
    .groupby(["model", "technique", "smell"], as_index=False)
    .agg(present_count=("is_present", "sum"))
)
per_configuration["evaluations"] = per_configuration.apply(
    lambda row: coverage.loc[row["model"], row["technique"]], axis=1
)
per_configuration["prevalence_pct"] = (
    100 * per_configuration["present_count"] / per_configuration["evaluations"]
)
display(per_configuration.sort_values("prevalence_pct", ascending=False)
        .groupby(["model", "technique"]).head(6).round(2).reset_index(drop=True))

In [ ]:
threshold = 0.80
mv3_runs = {}

for model in models:
    path = results_dir / f"{model}_mv3" / "responses.jsonl"

    with path.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            key = (model, item["prompt_id"])
            repetition = item["repetition"]
            runs = mv3_runs.setdefault(key, {})

            result = item.get("result")
            if isinstance(result, dict) and isinstance(result.get("smells"), dict):
                runs[repetition] = result

filtered_rows = []

for record in records:
    result = record["result"]

    if record["technique"] == "mv3":
        key = (record["model"], result["prompt_id"])
        runs = mv3_runs.get(key, {})
        assert len(runs) == 3, f"Expected 3 MV3 runs: {key}"

    for smell, annotation in result["smells"].items():
        if smell not in categories or annotation["label"] != "present":
            continue

        if record["technique"] == "mv3":
            votes = [
                run["smells"].get(smell, {})
                for run in runs.values()
            ]
            retained = sum(
                vote.get("label") == "present"
                and float(vote.get("confidence") or 0) >= threshold
                for vote in votes
            ) >= 2
        else:
            retained = float(annotation.get("confidence") or 0) >= threshold

        if retained:
            filtered_rows.append({
                "model": record["model"],
                "technique": record["technique"],
                "prompt_id": result["prompt_id"],
                "smell": smell,
                "category": categories[smell],
            })

df_filtered = pd.DataFrame(filtered_rows, columns=df_present.columns)

comparison = pd.DataFrame({
    "Without filter": df_present.groupby("model").size(),
    "Confidence ≥ 0.80": df_filtered.groupby("model").size(),
}).reindex(models).fillna(0).astype(int)

comparison["Reduction (%)"] = (
    (1 - comparison["Confidence ≥ 0.80"]
     / comparison["Without filter"].replace(0, float("nan"))) * 100
).round(1)

comparison

In [ ]:
confidence_rows = []

for record in records:
    result = record["result"]

    if record["technique"] == "mv3":
        runs = mv3_runs[(record["model"], result["prompt_id"])].values()
    else:
        runs = [result]

    for run in runs:
        for smell, annotation in run["smells"].items():
            confidence = annotation.get("confidence")

            if smell in categories and confidence is not None:
                confidence_rows.append({
                    "model": record["model"],
                    "confidence": float(confidence),
                })

confidence_mean = (
    pd.DataFrame(confidence_rows)
    .groupby("model")["confidence"]
    .mean()
    .reindex(models)
    .rename("Mean confidence")
    .round(3)
    .to_frame()
)

confidence_mean

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
per_configuration.to_csv(OUTPUT_DIR / "prevalence_by_configuration.csv", index=False)
coverage.to_csv(OUTPUT_DIR / "judge_coverage.csv")